In [1]:
import os
import time
from unstract.llmwhisperer import LLMWhispererClientV2

from dotenv import load_dotenv
load_dotenv()

client = LLMWhispererClientV2(base_url='https://llmwhisperer-api.eu-west.unstract.com/api/v2',
                              api_key=os.getenv('API_KEY'))

pdf_file_path = r'C:\Users\ALTHAAF\Documents\Code\Finance\CureForFinancialInsomnia\input\JohnKeells.pdf'
pdf_name = os.path.splitext(os.path.basename(pdf_file_path))[0]
result = client.whisper(file_path=pdf_file_path)

while True:
    status = client.whisper_status(whisper_hash=result['whisper_hash'])
    if status['status'] == 'processed':
        resultx = client.whisper_retrieve(whisper_hash=result['whisper_hash'])
        break

    time.sleep(5)

extracted_table = resultx['extraction']['result_text']

2026-07-20 18:15:08,843 - unstract.llmwhisperer.client_v2 - DEBUG - logging_level set to DEBUG
2026-07-20 18:15:08,844 - unstract.llmwhisperer.client_v2 - DEBUG - base_url set to https://llmwhisperer-api.eu-west.unstract.com/api/v2
2026-07-20 18:15:08,845 - unstract.llmwhisperer.client_v2 - DEBUG - whisper called
2026-07-20 18:15:08,846 - unstract.llmwhisperer.client_v2 - DEBUG - api_url: https://llmwhisperer-api.eu-west.unstract.com/api/v2/whisper
2026-07-20 18:15:08,847 - unstract.llmwhisperer.client_v2 - DEBUG - params: {'mode': 'form', 'output_mode': 'layout_preserving', 'page_seperator': '<<<', 'pages_to_extract': '', 'median_filter_size': 0, 'gaussian_blur_radius': 0, 'line_splitter_tolerance': 0.4, 'horizontal_stretch_factor': 1.0, 'mark_vertical_lines': False, 'mark_horizontal_lines': False, 'line_spitter_strategy': 'left-priority', 'add_line_nos': False, 'include_line_confidence': False, 'lang': 'eng', 'tag': 'default', 'filename': '', 'webhook_metadata': '', 'use_webhook': ''

In [2]:
print(extracted_table)



DECADE AT A GLANCE 

 In Rs. Mn 
 31 March                                  2026      2025      2024      2023      2022      2021      2020      2019       2018      2017 

OPERATING RESULTS 
Group revenue                            528,851   317,378   280,773   276,640   218,075   127,676   138,956    135,456   121,215   106,273 
EBIT                                      62,623    33,324    37,683    40,392    34,359     7,931    15,508     20,198    28,155    23,324 
Finance cost                             (24,935) (18,443)   (19,669) (17,803)     (7,035)   (4,669)   (3,166)   (2,722)     (521)     (436) 
Share of results of equity accounted      11,981    10,779    10,129     7,574     6,746     4,159     4,466      4,727     3,596     3,303 
  investees (net of tax) 
Profit before tax                         37,688    14,881    18,014    22,589    27,324     5,445    12,403     18,616    27,634    22,888 
Tax expense                              (15,606)   (7,957)   (5,886)    

In [3]:
import re
import os
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML

# 1. Helper function to parse individual numeric tokens
def parse_number_from_token(token):
    t = token.replace('−', '-').replace('–', '-').replace('—', '-')
    if re.match(r'^\d{4}[-/]\d{2,4}$', t):
        return None
    is_negative = False
    cleaned_t = t.strip()
    if cleaned_t.startswith('(') and cleaned_t.endswith(')'):
        is_negative = True
    elif cleaned_t.startswith('-'):
        is_negative = True
    cleaned = t.replace(',', '').replace('(', '').replace(')', '').replace('%', '').replace('$', '').replace('€', '').replace('£', '').replace('-', '').strip()
    if cleaned.replace('.', '', 1).isdigit():
        val = float(cleaned)
        return -val if is_negative else val
    elif cleaned == '' or cleaned.lower() in ['n/a', 'na', 'nil']:
        return 0.0
    return None

# 2. Candidate extraction helper function with line index association
def extract_candidates_from_text(raw_text):
    lines = raw_text.strip().split('\n')
    
    # Extract candidate years (matching formats like 2025/26, 2024-25, 2024)
    year_pattern = re.compile(r'\b\d{4}(?:/\d{2,4}|-\d{2,4})?\b')
    all_year_tokens = year_pattern.findall(raw_text)
    
    seen_years = set()
    candidate_years = []
    for yr in all_year_tokens:
        if yr not in seen_years:
            seen_years.add(yr)
            candidate_years.append(yr)
            
    # Extract candidate lines with their indices
    candidate_rows = [] # list of tuples: (display_name, line_idx)
    rows_preview_data = [] # preview for column guide
    for i, line in enumerate(lines):
        tokens = line.strip().split()
        if not tokens:
            continue
        
        nums = []
        text_parts = []
        for tok in tokens:
            val = parse_number_from_token(tok)
            if val is not None:
                nums.append(val)
            else:
                text_parts.append(tok)
                
        if nums:
            label = " ".join(text_parts).strip()
            nums_preview = ", ".join(str(n) for n in nums[:3])
            if len(nums) > 3:
                nums_preview += "..."
            
            display_lbl = label if label else "[No Label]"
            display_name = f"Row {i+1}: {display_lbl} | [{nums_preview}]"
            candidate_rows.append((display_name, i, label, tokens))
            
    return candidate_years, candidate_rows

# 3. Extract years and candidate rows
candidate_years, candidate_rows = extract_candidates_from_text(extracted_table)

# 4. Print the Column Index Guide for the first few rows
guide_html = "<h3 style='color:#202124; margin-bottom: 5px;'>Column Index Guide (First 4 rows with numbers)</h3>"
guide_html += "<table border='1' style='border-collapse:collapse; cellpadding:5px; font-family:monospace;'>"
guide_html += "<tr style='background-color:#f1f3f4;'><th>Row Source</th>"
max_cols_preview = 8
for c in range(max_cols_preview):
    guide_html += f"<th>Col {c}</th>"
guide_html += "</tr>"

for row in candidate_rows[:4]:
    disp_name, line_idx, lbl, tokens = row
    num_tokens = [t for t in tokens if parse_number_from_token(t) is not None]
    guide_html += f"<tr><td><b>Row {line_idx+1}:</b> {lbl if lbl else '[No Label]'}</td>"
    for c in range(max_cols_preview):
        val = num_tokens[c] if c < len(num_tokens) else ""
        guide_html += f"<td>{val}</td>"
    guide_html += "</tr>"
guide_html += "</table>"
guide_html += "<p style='font-size:12px; color:#5f6368; margin-top:5px;'>Use this guide to identify which column indices contain the absolute metrics for your target years (e.g. if you see percentages between years, choose indices like 0, 2, 4...).</p>"

# 5. Target Keys configuration
target_keys = {
    "revenue": "Revenue",
    "pbt": "Profit/(Loss) before taxation",
    "total_equity": "Total equity",
    "total_borrowings": "Total borrowings",
    "current_assets": "Current assets",
    "total_assets": "Total assets employed",
    "current_ratio": "Current ratio (No. of times)",
    "quick_ratio": "Quick asset ratio",
    "gearing_ratio": "Gearing ratio (%)"
}

# 6. Create interactive widgets
default_years = ", ".join(candidate_years[:6])
years_input = widgets.Text(
    value=default_years,
    placeholder='e.g., 2025/26, 2024/25, 2023/24...',
    description='Target Years (comma separated):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='60%')
)

col_indices_input = widgets.Text(
    value="",
    placeholder='e.g., 0, 2, 4, 6, 8, 10 (Leave blank for consecutive 0, 1, 2...)',
    description='Column Indices to Extract (comma separated):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='60%')
)

dropdowns = {}
# Use tuple format (label_to_display, integer_line_index) for ipywidgets.Dropdown
options = [("-- Select Row --", -1)] + [(row[0], row[1]) for row in candidate_rows]

for json_key, default_lbl in target_keys.items():
    selected_line_idx = -1
    for row in candidate_rows:
        disp_name, line_idx, lbl, tokens = row
        if default_lbl.lower() in lbl.lower() or lbl.lower() in default_lbl.lower():
            selected_line_idx = line_idx
            break
            
    if selected_line_idx == -1:
        if json_key == "revenue":
            for row in candidate_rows:
                disp_name, line_idx, lbl, tokens = row
                if "revenue" in lbl.lower() or "turnover" in lbl.lower() or "income" in lbl.lower():
                    if "reserve" not in lbl.lower():
                        selected_line_idx = line_idx
                        break
        elif json_key == "pbt":
            for row in candidate_rows:
                disp_name, line_idx, lbl, tokens = row
                if "before taxation" in lbl.lower() or "before tax" in lbl.lower() or "pbt" in lbl.lower():
                    selected_line_idx = line_idx
                    break
        elif json_key == "total_assets":
            for row in candidate_rows:
                disp_name, line_idx, lbl, tokens = row
                if "total assets" in lbl.lower():
                    selected_line_idx = line_idx
                    break
                    
    dropdowns[json_key] = widgets.Dropdown(
        options=options,
        value=selected_line_idx if any(val == selected_line_idx for _, val in options) else -1,
        description=f"{json_key} (Default: '{default_lbl}'):",
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='75%')
    )

dropdown_list = [dropdowns[k] for k in target_keys.keys()]

btn_extract = widgets.Button(
    description='Run Extraction',
    button_style='success',
    tooltip='Click to parse metrics and run financial analysis',
    icon='check',
    layout=widgets.Layout(margin='15px 0px 0px 0px')
)

output_area = widgets.Output()

# 7. Define Button Click Callback
def on_btn_clicked(b):
    with output_area:
        output_area.clear_output()
        
        # 1. Parse target years
        target_yrs = [yr.strip() for yr in years_input.value.split(",") if yr.strip()]
        if not target_yrs:
            print("❌ Please specify at least one target year.")
            return
            
        # 2. Parse column indices to extract
        try:
            col_indices = [int(idx.strip()) for idx in col_indices_input.value.split(",") if idx.strip()]
        except Exception:
            print("❌ Invalid Column Indices format. Use comma separated numbers (e.g. 0, 2, 4).")
            return
            
        if not col_indices:
            col_indices = list(range(len(target_yrs)))
            
        # 3. Get selected hooks mapping: JSON key -> Line Index (integer)
        selected_hooks = {}
        for json_key, dropdown in dropdowns.items():
            if dropdown.value != -1:
                selected_hooks[json_key] = dropdown.value
                    
        if not selected_hooks:
            print("❌ Please map at least one metric.")
            return
            
        print("⏳ Extracting data from LLMWhisperer table using line indices...")
        
        # 4. Perform extraction
        data_store = {year: {} for year in target_yrs}
        lines = extracted_table.strip().split('\n')
        
        for json_key, line_idx in selected_hooks.items():
            if line_idx >= len(lines):
                continue
            line = lines[line_idx]
            tokens = line.strip().split()
            
            clean_numbers = []
            for tok in tokens:
                val = parse_number_from_token(tok)
                if val is not None:
                    clean_numbers.append(val)
                    
            # Apply column indices mapping
            for year_idx, year in enumerate(target_yrs):
                if year_idx < len(col_indices):
                    target_col_idx = col_indices[year_idx]
                    if target_col_idx < len(clean_numbers):
                        data_store[year][json_key] = clean_numbers[target_col_idx]
                            
        # Ensure all mapped keys exist in each year
        for year in data_store:
            for k in target_keys.keys():
                if k not in data_store[year]:
                    data_store[year][k] = 0.0
                    
        # 5. Convert to DataFrame
        global df, analysis_df, original_df, analysis_results_df, trimmed_analysis_df
        
        df = pd.DataFrame(data_store).T
        df.index.name = 'Year'
        df = df.reset_index()
        
        print("✅ Raw Data Extracted:")
        display(df)
        
        # 6. Run analysis calculations
        try:
            analysis_df = df.copy()
            required_cols = ['revenue', 'pbt', 'total_equity', 'total_borrowings', 'current_assets', 'total_assets', 'current_ratio', 'quick_ratio', 'gearing_ratio']
            for col in required_cols:
                if col not in analysis_df.columns:
                    analysis_df[col] = 0.0
                else:
                    analysis_df[col] = pd.to_numeric(analysis_df[col]).fillna(0.0)
                    
            # Use last row as baseline year
            base_year_idx = len(analysis_df) - 1
            if base_year_idx >= 0:
                base_year_row = analysis_df.loc[base_year_idx]
            else:
                base_year_row = pd.Series(0.0, index=analysis_df.columns)
                
            core_metrics = ['revenue', 'pbt', 'total_equity', 'total_borrowings', 'current_assets', 'total_assets']
            
            # Helper variables
            safe_current_ratio = analysis_df['current_ratio'].replace(0, 1.0)
            current_liabs = analysis_df['current_assets'] / safe_current_ratio
            analysis_df['net_working_capital_abs'] = analysis_df['current_assets'] - current_liabs
            
            # 1. Horizontal Growth
            for metric in core_metrics:
                analysis_df[f'{metric}_horizontal_growth_%'] = analysis_df[metric].pct_change(-1) * 100
                base_val = base_year_row[metric] if base_year_row[metric] != 0 else 1.0
                analysis_df[f'{metric}_trend_index'] = (analysis_df[metric] / base_val) * 100
                
            # 2. Vertical Analysis
            safe_total_assets = analysis_df['total_assets'].replace(0, 1.0)
            safe_revenue = analysis_df['revenue'].replace(0, 1.0)
            analysis_df['current_assets_vertical_%'] = (analysis_df['current_assets'] / safe_total_assets) * 100
            analysis_df['total_equity_vertical_%'] = (analysis_df['total_equity'] / safe_total_assets) * 100
            analysis_df['borrowings_vertical_%'] = (analysis_df['total_borrowings'] / safe_total_assets) * 100
            analysis_df['pbt_vertical_margin_%'] = (analysis_df['pbt'] / safe_revenue) * 100
            
            # 3. Ratios
            analysis_df['net_profit_margin_%'] = (analysis_df['pbt'] / safe_revenue) * 100
            analysis_df['return_on_assets_roa_%'] = (analysis_df['pbt'] / safe_total_assets) * 100
            safe_total_equity = analysis_df['total_equity'].replace(0, 1.0)
            analysis_df['return_on_equity_roe_%'] = (analysis_df['pbt'] / safe_total_equity) * 100
            analysis_df['working_capital_to_assets_%'] = (analysis_df['net_working_capital_abs'] / safe_total_assets) * 100
            analysis_df['debt_to_equity_ratio'] = analysis_df['total_borrowings'] / safe_total_equity
            analysis_df['equity_to_total_assets_%'] = (analysis_df['total_equity'] / safe_total_assets) * 100
            
            # 4. Altman Z-Score
            analysis_df['Z_X1'] = analysis_df['net_working_capital_abs'] / safe_total_assets
            analysis_df['Z_X2'] = analysis_df['total_equity'] / safe_total_assets
            analysis_df['Z_X3'] = analysis_df['pbt'] / safe_total_assets
            safe_total_borrowings = analysis_df['total_borrowings'].replace(0, 1.0)
            analysis_df['Z_X4'] = analysis_df['total_equity'] / safe_total_borrowings
            analysis_df['Z_X5'] = analysis_df['revenue'] / safe_total_assets
            
            analysis_df['altman_z_score'] = (1.2 * analysis_df['Z_X1']) + (1.4 * analysis_df['Z_X2']) + \
                                           (3.3 * analysis_df['Z_X3']) + (0.6 * analysis_df['Z_X4']) + \
                                           (0.999 * analysis_df['Z_X5'])
                                           
            def categorize_z_zone(z):
                if z > 2.90: return "Safe Zone"
                elif z < 1.23: return "Distress Zone"
                return "Grey Zone"
                
            analysis_df['altman_risk_classification'] = analysis_df['altman_z_score'].apply(categorize_z_zone)
            analysis_df = analysis_df.round(2)
            
            trim_len = max(1, len(target_yrs) - 1)
            trimmed_analysis_df = analysis_df.iloc[:trim_len].copy()
            
            original_cols = ['Year', 'revenue', 'pbt', 'total_equity', 'total_borrowings', 
                             'current_assets', 'total_assets', 'gearing_ratio', 'current_ratio', 'quick_ratio']
            for col in original_cols:
                if col not in analysis_df.columns:
                    analysis_df[col] = 0.0
            original_df = analysis_df[original_cols].copy()
            
            analysis_cols = [col for col in analysis_df.columns if col not in original_cols or col == 'Year']
            analysis_results_df = analysis_df[analysis_cols].iloc[:trim_len].copy()
            
            print("📈 Financial Analysis Calculated successfully!")
            print("\n--- Original Metrics ---")
            display(original_df)
            print("\n--- Analysis Results ---")
            display(analysis_results_df)
            
            # 7. Save to Excel
            excel_name = f"{pdf_name}_analysis.xlsx"
            folder_path = "output"
            if not os.path.exists(folder_path):
                os.makedirs(folder_path)
            excel_path = os.path.join(folder_path, excel_name)
            
            with pd.ExcelWriter(excel_path) as writer:
                original_df.to_excel(writer, sheet_name='Original Metrics', index=False)
                analysis_results_df.to_excel(writer, sheet_name='Analysis Results', index=False)
                
            print(f"\n💾 Excel file '{excel_name}' created successfully in '{folder_path}' folder.")
            
        except Exception as ex:
            print(f"❌ Error during calculation or Excel saving: {ex}")
            import traceback
            traceback.print_exc()

btn_extract.on_click(on_btn_clicked)

# Render layout
display(HTML("<h2 style='color:#1a73e8; margin-top:0px;'>Interactive Financial Parser Mapping</h2>"))
display(HTML(guide_html))
display(years_input)
display(col_indices_input)
display(widgets.VBox(dropdown_list))
display(btn_extract)
display(output_area)


Row Source,Col 0,Col 1,Col 2,Col 3,Col 4,Col 5,Col 6,Col 7
Row 4: March,31,2026,2025,2024,2023,2022,2021,2020
Row 7: Group revenue,"528,851","317,378","280,773","276,640","218,075","127,676","138,956","135,456"
Row 8: EBIT,"62,623","33,324","37,683","40,392","34,359","7,931","15,508","20,198"
Row 9: Finance cost,"(24,935)","(18,443)","(19,669)","(17,803)","(7,035)","(4,669)","(3,166)","(2,722)"


Text(value='2026, 2025, 2024, 2023, 2022, 2021', description='Target Years (comma separated):', layout=Layout(…

Text(value='', description='Column Indices to Extract (comma separated):', layout=Layout(width='60%'), placeho…

Button(button_style='success', description='Run Extraction', icon='check', layout=Layout(margin='15px 0px 0px …

Output()

In [ ]:
df

In [ ]:
original_df

In [ ]:
analysis_results_df

In [ ]:
trimmed_analysis_df